In [2]:
import numpy as np
from utils.defines import *
import os
import json

In [3]:
# Load images, labels, predictions
images = []
labels = []
predictions = []
label_dict = {} # label: name of label, i.e., {0: dog, 1: cat}

In [ ]:
# Chat with our AI assistant, get visual attributes
# Alternatively, you can propose the visual attributes by yourself. Then skip this step.
from utils.gpt import ChatBot
# api_key = "" # Your GPT api key
# ChatBot = ChatBot(api_key)
ChatBot = ChatBot()
ChatBot.chat() # chat with our AI assistent, you can stop when you satisfied with the attributes in the form

AttributeError: module 'gradio' has no attribute 'ChatInterface'

In [6]:
# Define train_idx, valid_idx, unlabel_idx, data split
train_idx = []
valid_idx = []
split = [UNLABELED for i in range(len(images))]
for i in train_idx:
    split[i] = TRAIN
for i in valid_idx:
    split[i] = VALID

In [7]:
from utils.gpt import get_des_question, get_binary_question,get_des_prompt, get_type
from processData.Corpus import emptyAttribute
import json
attributes = ChatBot.attributes # a list of attributes: ["attribute1", "attribute2", "attribute3"]
attribute_dict = {}
main_object = ""
for name in attributes:
    attribute_dict[name] = emptyAttribute()
    data_type = get_type(main_object,name)
    attribute_dict[name]['type'] = data_type
    if data_type == 'binary':
        attribute_dict[name]['question'] = get_binary_question(main_object,name)
    else:
        attribute_dict[name]['question'] = get_des_question(main_object,name)
        attribute_dict[name]['prompt'] = [get_des_prompt(main_object,name),]
# Although GPT is amazing, it is better to double-check the corpus base
for key, content in attribute_dict.items():
    print(key, content)
f = open("exampleData/celebA/corpus.json" ,'w')
f.write(json.dumps(attribute_dict,indent=4))

AttributeError: module 'cv2.gapi.wip.draw' has no attribute 'Text'

In [8]:
# Find potential attribute value for this task
from processData.Corpus import AttributeCorpus
corpus = AttributeCorpus(datas=images,attribute_dict="exampleData/corpus_base.json")
corpus.auto_corpus()
corpus.export_json(file="exampleData/corpus.json")

AttributeError: module 'cv2.gapi.wip.draw' has no attribute 'Text'

In [ ]:
# Get the attribute of every data
from processData.AttributeSelection import AttributeSelection
AS = AttributeSelection(images, labels, corpus="exampleData/corpus.json",label_names_dict=label_dict)
AS.match_description_to_data()
AS.save_attributes("exampleData/attribute.json")

In [ ]:
# User Interface, require 2D feature as input
from interaction.interaction import Interaction
embeddings = np.load("") # if not 2D, we will run TSNE automatically
interaction = Interaction(
   datas=images,
   labels=labels,
   predictions=predictions,
   embeddings=embeddings,
   tags=split,
   attributes_dict_path="exampleData/attribute.json"
)
interaction.interaction()
 

In [ ]:
# Diagnose 
from processData.Diagnose import ModelDiagnose
MD = ModelDiagnose(labels, predictions, split, "exampleData/attribute.json")
MD.detect_failure_by_label()
MD.detect_prediction_correlation()
MD.detect_failure_prediction_correlation()
MD.detect_failure_in_attribute_combination()

In [ ]:
# Repair
selected_unlabeled_idx = MD.unlabel_selection()
generation_attribute_dict = MD.date_generation()